In [1]:
import requests
import pandas as pd
import json

# Overpass API-Abfrage für Tankstellen in der Schweiz
overpass_url = "http://overpass-api.de/api/interpreter"
overpass_query = """
[out:json];
area["ISO3166-1"="CH"][admin_level=2];
node["amenity"="fuel"](area);
out body;
>;
out skel qt;
"""
response = requests.get(overpass_url, params={'data': overpass_query})
data = response.json()

# Speichern der Daten in .json-Format
with open('petrol_stations_switzerland.json', 'w') as json_file:
    json.dump(data, json_file)

# Extrahieren der relevanten Informationen und Speichern in .csv-Format
elements = data['elements']
rows = []
for element in elements:
    if 'tags' in element and 'brand' in element['tags']:
        rows.append({
            'id': element['id'],
            'lat': element['lat'],
            'lon': element['lon'],
            'brand': element['tags']['brand']
        })

df = pd.DataFrame(rows)
df.to_csv('petrol_stations_switzerland.csv', index=False)

 Anzahl der Tankstellen pro Betreiber zählen und die 5 größten Betreiber identifizieren

In [2]:
# Laden der Daten
df = pd.read_csv('petrol_stations_switzerland.csv')

# Anzahl der Tankstellen pro Betreiber zählen
pivot_table = df.pivot_table(index='brand', aggfunc='size').reset_index(name='count')

# Sortieren und die 5 größten Betreiber identifizieren
top_5_operators = pivot_table.sort_values(by='count', ascending=False).head(5)
print(top_5_operators)

     brand  count
7     Avia    284
4   Agrola    191
10      BP    170
49  Migrol    167
74   Shell    159


Karte erstellen und Tankstellen visualisieren

In [3]:
import folium

# Erstellen einer Karte
m = folium.Map(location=[46.8182, 8.2275], zoom_start=8)

# Tankstellen zur Karte hinzufügen
for _, row in df.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=row['brand']
    ).add_to(m)

# Speichern der Karte
m.save('petrol_stations_switzerland_map.html')